# Model Benchmark - Detailed Report

**Models:**
- Qwen/Qwen3-8B, Qwen/Qwen3-4B
- vinai/PhoGPT-4B-Chat
- deepseek-ai/deepseek-llm-7b-chat
- keepitreal/vietnamese-sbert

**Output:**
- Detailed JSON report for each model with per-question analysis
- Including: model name, model choice, model response, correctness, question details

In [1]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

In [2]:
# Cell 2: Imports & GPU Check
import os
import json
import re
import gc
import time
from datetime import datetime
from typing import List, Dict, Optional, Tuple
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

print("="*60)
print("GPU STATUS")
print("="*60)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.1f}GB")
    DEVICE = 'cuda'
else:
    print("No GPU")
    DEVICE = 'cpu'
print("="*60)

2025-12-12 00:25:45.746609: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765499145.932263      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765499145.986017      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

GPU STATUS
GPU 0: Tesla T4 | 14.7GB
GPU 1: Tesla T4 | 14.7GB


In [3]:
# Cell 3: Configuration - INCREASED CONTEXT
DATA_PATH = "/kaggle/input/question_1000.json"  # Kaggle
# DATA_PATH = "D:/KLTN/data/question_1000.json"  # Local

OUTPUT_DIR = "./results"
BATCH_SIZE = 4  # Reduced for more stable generation with longer context

# INCREASED CONTEXT - Tăng context đầu vào và đầu ra
MAX_INPUT_LENGTH = 1024  # Tăng từ 1024
MAX_NEW_TOKENS = 128     # Tăng từ 20 để có response đầy đủ hơn

MAX_QUESTIONS = None  # Set to limit

MODELS = [
    {"name": "Qwen/Qwen3-4B", "type": "causal"},
    {"name": "Qwen/Qwen3-8B", "type": "causal"},
    {"name": "vinai/PhoGPT-4B-Chat", "type": "causal"},
    {"name": "deepseek-ai/deepseek-llm-7b-chat", "type": "causal"},
    {"name": "keepitreal/vietnamese-sbert", "type": "embed"},
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config: batch={BATCH_SIZE}, max_input={MAX_INPUT_LENGTH}, max_output={MAX_NEW_TOKENS}")

Config: batch=4, max_input=1024, max_output=128


In [4]:
# Cell 4: Load Data - Load both multiple_choice AND true_false
def load_data(path: str):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Load MCQ from 'multiple_choice' key
    mcq_raw = data.get('multiple_choice', [])
    print(f"Found 'multiple_choice': {len(mcq_raw)} questions")
    
    # Load T/F from 'true_false' key
    tf_raw = data.get('true_false', [])
    print(f"Found 'true_false': {len(tf_raw)} questions")
    
    mcq, tf = [], []
    
    # Process MCQ
    for idx, q in enumerate(mcq_raw):
        opts = q.get('options', [])
        if not opts:
            continue
        correct_idx = next((i for i, o in enumerate(opts) if o.get('isCorrect')), 0)
        mcq.append({
            'id': f'mcq_{idx}',
            'q': q['question'],
            'opts': [o['answer'] for o in opts],
            'opts_full': opts,  # Keep full options for detailed report
            'ans': chr(65 + correct_idx),
            'ans_text': opts[correct_idx]['answer'],
            'idx': correct_idx,
            'type': 'MCQ'
        })
    
    # Process True/False
    for idx, q in enumerate(tf_raw):
        opts = q.get('options', [])
        if not opts:
            continue
        correct_idx = next((i for i, o in enumerate(opts) if o.get('isCorrect')), 0)
        correct_answer = opts[correct_idx]['answer']
        # Normalize answer
        is_true = correct_answer.lower() in ['dung', 'true', 'correct', 'đúng'] or 'ú' in correct_answer.lower()[:4]
        tf.append({
            'id': f'tf_{idx}',
            'q': q['question'],
            'opts': [o['answer'] for o in opts],
            'opts_full': opts,
            'ans': 'Đúng' if is_true else 'Sai',
            'ans_text': correct_answer,
            'idx': correct_idx,
            'type': 'T/F'
        })
    
    return mcq, tf

mcq_data, tf_data = load_data(DATA_PATH)

if MAX_QUESTIONS:
    limit = MAX_QUESTIONS // 2
    mcq_data = mcq_data[:limit]
    tf_data = tf_data[:limit]

print(f"\nLoaded for evaluation:")
print(f"  - MCQ: {len(mcq_data)}")
print(f"  - True/False: {len(tf_data)}")
print(f"  - Total: {len(mcq_data) + len(tf_data)}")

Found 'multiple_choice': 512 questions
Found 'true_false': 488 questions

Loaded for evaluation:
  - MCQ: 512
  - True/False: 488
  - Total: 1000


In [5]:
# Cell 5: Enhanced Prompts with more context
def make_mcq_prompt(q, opts):
    """Create MCQ prompt with more context and clearer instructions"""
    o = '\n'.join([f"{chr(65+i)}. {x}" for i, x in enumerate(opts)])
    prompt = f"""Bạn là một chuyên gia về lịch sử Việt Nam và thế giới. Hãy đọc kỹ câu hỏi và các lựa chọn sau, sau đó chọn đáp án đúng nhất.

Câu hỏi: {q}

Các lựa chọn:
{o}

Hãy phân tích ngắn gọn và đưa ra đáp án cuối cùng. Đáp án của bạn phải là MỘT trong các chữ cái A, B, C, hoặc D.

Đáp án:"""
    return prompt

def make_tf_prompt(q):
    """Create True/False prompt with more context"""
    prompt = f"""Bạn là một chuyên gia về lịch sử Việt Nam và thế giới. Hãy đọc kỹ phát biểu sau và xác định xem nó ĐÚNG hay SAI.

Phát biểu: {q}

Hãy phân tích ngắn gọn và đưa ra kết luận. Trả lời bằng "Đúng" nếu phát biểu đúng, hoặc "Sai" nếu phát biểu sai.

Kết luận:"""
    return prompt

def extract_mcq(text):
    """Extract MCQ answer with improved parsing"""
    if not text:
        return None, "Empty response"
    
    text_upper = text.strip().upper()
    
    # Try multiple patterns
    patterns = [
        r'ĐÁP ÁN[:\s]*([ABCD])',
        r'DAP AN[:\s]*([ABCD])',
        r'CHỌN[:\s]*([ABCD])',
        r'LÀ[:\s]*([ABCD])',
        r'\b([ABCD])\.$',
        r'\b([ABCD])\b(?!\w)',
    ]
    
    for pattern in patterns:
        m = re.search(pattern, text_upper)
        if m:
            return m.group(1), "Pattern matched"
    
    # Fallback: first letter A-D
    for c in text_upper:
        if c in 'ABCD': 
            return c, "First letter matched"
    
    return None, "No answer found"

def extract_tf(text):
    """Extract True/False answer with improved parsing"""
    if not text:
        return None, "Empty response"
    
    t = text.lower()
    
    # Check for True indicators
    true_indicators = ['đúng', 'dung', 'true', 'correct', 'chính xác', 'yes', 'có']
    false_indicators = ['sai', 'false', 'wrong', 'không đúng', 'incorrect', 'no', 'không']
    
    for indicator in true_indicators:
        if indicator in t:
            # Make sure it's not negated
            pattern = rf'(?<!không )(?<!chưa ){re.escape(indicator)}'
            if re.search(pattern, t):
                return 'Đúng', f"Matched: {indicator}"
    
    for indicator in false_indicators:
        if indicator in t:
            return 'Sai', f"Matched: {indicator}"
    
    return None, "No answer found"

print("Prompts ready")

Prompts ready


In [6]:
# Cell 6: Model Loading
def get_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

def load_model(name: str, model_type: str):
    if model_type == 'embed':
        return SentenceTransformer(name, device=DEVICE), None
    
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True, padding_side='left')
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        name,
        quantization_config=get_bnb_config(),
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model.eval()
    return model, tok

def aggressive_cleanup():
    gc.collect()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()
        gc.collect()

def unload(model, tok):
    if model is not None:
        try:
            model.cpu()
        except:
            pass
        del model
    if tok is not None:
        del tok
    aggressive_cleanup()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            used = torch.cuda.memory_allocated(i) / 1024**3
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f"   GPU {i} memory: {used:.1f}/{total:.1f} GB")

print("Model loader ready")

Model loader ready


In [7]:
# Cell 7: Batched Inference with Detailed Tracking
@torch.inference_mode()
def batch_generate(model, tokenizer, prompts: List[str], max_tokens: int = MAX_NEW_TOKENS) -> List[str]:
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(model.device)
    
    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs.get('attention_mask'),
        max_new_tokens=max_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    
    responses = []
    input_len = inputs['input_ids'].shape[1]
    for out in outputs:
        resp = tokenizer.decode(out[input_len:], skip_special_tokens=True)
        responses.append(resp.strip())
    
    del inputs, outputs
    return responses

def eval_causal_batched_detailed(model, tok, questions, q_type, batch_size=BATCH_SIZE):
    """Evaluate with detailed tracking per question"""
    if not questions:
        return []
    
    detailed_results = []
    is_mcq = (q_type == 'MCQ')
    
    prompts = []
    for q in questions:
        if is_mcq:
            prompts.append(make_mcq_prompt(q['q'], q['opts']))
        else:
            prompts.append(make_tf_prompt(q['q']))
    
    for i in tqdm(range(0, len(prompts), batch_size), desc=f"   {q_type}", leave=False):
        batch_prompts = prompts[i:i+batch_size]
        batch_questions = questions[i:i+batch_size]
        
        try:
            responses = batch_generate(model, tok, batch_prompts, MAX_NEW_TOKENS)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"   OOM, trying one by one...")
                torch.cuda.empty_cache()
                responses = []
                for p in batch_prompts:
                    try:
                        r = batch_generate(model, tok, [p], MAX_NEW_TOKENS)
                        responses.extend(r)
                    except:
                        responses.append("[ERROR: OOM]")
            else:
                raise e
        
        for idx, (resp, q, prompt) in enumerate(zip(responses, batch_questions, batch_prompts)):
            if is_mcq:
                pred, extraction_note = extract_mcq(resp)
            else:
                pred, extraction_note = extract_tf(resp)
            
            is_correct = pred == q['ans']
            
            # Create detailed result
            detailed_result = {
                # Question info
                'question_id': q['id'],
                'question_type': q['type'],
                'question_text': q['q'],
                'options': q['opts'],
                'correct_answer': q['ans'],
                'correct_answer_text': q['ans_text'],
                
                # Model response
                'prompt_used': prompt,
                'model_response_full': resp,
                'model_choice': pred,
                'extraction_method': extraction_note,
                
                # Evaluation
                'is_correct': is_correct,
                'response_length': len(resp),
            }
            
            detailed_results.append(detailed_result)
    
    return detailed_results

def eval_embed_detailed(model, questions, q_type):
    """Evaluate embedding model with detailed tracking"""
    if not questions:
        return []
    
    detailed_results = []
    is_mcq = (q_type == 'MCQ')
    
    all_q = [q['q'] for q in questions]
    q_embs = model.encode(all_q, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    
    for i, q in tqdm(enumerate(questions), total=len(questions), desc=f"   {q_type}", leave=False):
        opt_embs = model.encode(q['opts'], normalize_embeddings=True)
        sims = q_embs[i] @ opt_embs.T
        pred_idx = int(sims.argmax())
        
        if is_mcq:
            pred = chr(65 + pred_idx)
        else:
            opt = q['opts'][pred_idx].lower()
            pred = 'Đúng' if any(x in opt for x in ['đúng', 'dung', 'true']) else 'Sai'
        
        is_correct = pred == q['ans']
        
        # Similarity scores for each option
        similarity_scores = {q['opts'][j]: float(sims[j]) for j in range(len(q['opts']))}
        
        detailed_result = {
            'question_id': q['id'],
            'question_type': q['type'],
            'question_text': q['q'],
            'options': q['opts'],
            'correct_answer': q['ans'],
            'correct_answer_text': q['ans_text'],
            
            'prompt_used': f"[Embedding similarity] {q['q']}",
            'model_response_full': f"Similarity-based selection: {pred}",
            'model_choice': pred,
            'extraction_method': 'Embedding similarity',
            'similarity_scores': similarity_scores,
            
            'is_correct': is_correct,
            'response_length': 0,
        }
        
        detailed_results.append(detailed_result)
    
    return detailed_results

print("Inference ready")

Inference ready


In [8]:
# Cell 8: Main Evaluation with Detailed Reports
def calculate_metrics(results: List[Dict]) -> Dict:
    """Calculate detailed metrics from results"""
    if not results:
        return {}
    
    total = len(results)
    correct = sum(1 for r in results if r['is_correct'])
    
    # Breakdown by question type
    mcq_results = [r for r in results if r['question_type'] == 'MCQ']
    tf_results = [r for r in results if r['question_type'] == 'T/F']
    
    mcq_correct = sum(1 for r in mcq_results if r['is_correct'])
    tf_correct = sum(1 for r in tf_results if r['is_correct'])
    
    # Response analysis
    avg_response_length = sum(r['response_length'] for r in results) / total if total > 0 else 0
    null_predictions = sum(1 for r in results if r['model_choice'] is None)
    
    return {
        'total_questions': total,
        'correct_answers': correct,
        'accuracy_percent': round(correct / total * 100, 2) if total > 0 else 0,
        
        'mcq_total': len(mcq_results),
        'mcq_correct': mcq_correct,
        'mcq_accuracy_percent': round(mcq_correct / len(mcq_results) * 100, 2) if mcq_results else 0,
        
        'tf_total': len(tf_results),
        'tf_correct': tf_correct,
        'tf_accuracy_percent': round(tf_correct / len(tf_results) * 100, 2) if tf_results else 0,
        
        'average_response_length': round(avg_response_length, 2),
        'null_predictions': null_predictions,
        'null_prediction_rate_percent': round(null_predictions / total * 100, 2) if total > 0 else 0,
    }

def evaluate_model_detailed(cfg, mcq, tf):
    """Evaluate model and return detailed results"""
    name, mtype = cfg['name'], cfg['type']
    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"{'='*60}")
    
    aggressive_cleanup()
    
    t0 = time.time()
    try:
        model, tok = load_model(name, mtype)
    except Exception as e:
        print(f"   Load failed: {e}")
        aggressive_cleanup()
        return None
    
    load_time = time.time() - t0
    print(f"   Loaded in {load_time:.1f}s")
    
    t1 = time.time()
    try:
        if mtype == 'embed':
            mcq_res = eval_embed_detailed(model, mcq, 'MCQ')
            tf_res = eval_embed_detailed(model, tf, 'T/F')
        else:
            mcq_res = eval_causal_batched_detailed(model, tok, mcq, 'MCQ', BATCH_SIZE)
            tf_res = eval_causal_batched_detailed(model, tok, tf, 'T/F', BATCH_SIZE)
    except Exception as e:
        print(f"   Evaluation failed: {e}")
        import traceback
        traceback.print_exc()
        unload(model, tok)
        return None
    
    eval_time = time.time() - t1
    total_time = time.time() - t0
    
    # Combine all results
    all_results = mcq_res + tf_res
    
    # Calculate metrics
    metrics = calculate_metrics(all_results)
    
    # Create model report
    model_report = {
        'model_name': name,
        'model_type': mtype,
        'evaluation_timestamp': datetime.now().isoformat(),
        
        # Configuration
        'config': {
            'max_input_length': MAX_INPUT_LENGTH,
            'max_new_tokens': MAX_NEW_TOKENS,
            'batch_size': BATCH_SIZE,
        },
        
        # Timing
        'timing': {
            'load_time_seconds': round(load_time, 2),
            'eval_time_seconds': round(eval_time, 2),
            'total_time_seconds': round(total_time, 2),
            'questions_per_second': round(len(all_results) / eval_time, 2) if eval_time > 0 else 0,
        },
        
        # Metrics
        'metrics': metrics,
        
        # Detailed per-question results
        'detailed_results': all_results,
    }
    
    # Print summary
    print(f"\n   Results Summary:")
    print(f"   - Overall Accuracy: {metrics['accuracy_percent']:.2f}% ({metrics['correct_answers']}/{metrics['total_questions']})")
    print(f"   - MCQ Accuracy: {metrics['mcq_accuracy_percent']:.2f}% ({metrics['mcq_correct']}/{metrics['mcq_total']})")
    print(f"   - T/F Accuracy: {metrics['tf_accuracy_percent']:.2f}% ({metrics['tf_correct']}/{metrics['tf_total']})")
    print(f"   - Null predictions: {metrics['null_predictions']} ({metrics['null_prediction_rate_percent']:.2f}%)")
    print(f"   - Avg response length: {metrics['average_response_length']:.0f} chars")
    print(f"   - Speed: {model_report['timing']['questions_per_second']:.2f} q/s")
    
    unload(model, tok)
    return model_report

print("Evaluator ready")

Evaluator ready


In [9]:
# Cell 9: Run Benchmark
print("\n" + "#"*70)
print("#" + " "*20 + "BENCHMARK START" + " "*33 + "#")
print("#"*70 + "\n")

print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
print(f"Questions: {len(mcq_data)} MCQ + {len(tf_data)} T/F = {len(mcq_data) + len(tf_data)} total")
print(f"Models: {len(MODELS)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max input length: {MAX_INPUT_LENGTH}")
print(f"Max output tokens: {MAX_NEW_TOKENS}")

all_model_reports = []
benchmark_start_time = time.time()

for i, cfg in enumerate(MODELS):
    print(f"\n[{i+1}/{len(MODELS)}] {cfg['name']}")
    
    try:
        report = evaluate_model_detailed(cfg, mcq_data, tf_data)
        if report:
            all_model_reports.append(report)
            
            # Save individual model report
            model_safe_name = cfg['name'].replace('/', '_').replace('-', '_')
            model_report_path = os.path.join(OUTPUT_DIR, f"detailed_report_{model_safe_name}.json")
            with open(model_report_path, 'w', encoding='utf-8') as f:
                json.dump(report, f, ensure_ascii=False, indent=2)
            print(f"   Saved: {model_report_path}")
    except Exception as e:
        print(f"   Error: {e}")
        import traceback
        traceback.print_exc()
        aggressive_cleanup()

benchmark_end_time = time.time()
total_benchmark_time = benchmark_end_time - benchmark_start_time

print(f"\nBenchmark completed in {total_benchmark_time/60:.1f} minutes")


######################################################################
#                    BENCHMARK START                                 #
######################################################################

Start: 00:26:07
Questions: 512 MCQ + 488 T/F = 1000 total
Models: 5
Batch size: 4
Max input length: 1024
Max output tokens: 128

[1/5] Qwen/Qwen3-4B

Model: Qwen/Qwen3-4B


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   Loaded in 8.5s


   MCQ:   0%|          | 0/512 [00:00<?, ?it/s]

   T/F:   0%|          | 0/488 [00:00<?, ?it/s]


   Results Summary:
   - Overall Accuracy: 39.60% (396/1000)
   - MCQ Accuracy: 29.69% (152/512)
   - T/F Accuracy: 50.00% (244/488)
   - Null predictions: 0 (0.00%)
   - Avg response length: 0 chars
   - Speed: 45.42 q/s
   GPU 0 memory: 0.0/14.7 GB
   GPU 1 memory: 0.0/14.7 GB
   Saved: ./results/detailed_report_keepitreal_vietnamese_sbert.json

Benchmark completed in 319.3 minutes


In [10]:
# Cell 10: Generate Summary Report
def generate_summary_report(all_reports: List[Dict]) -> Dict:
    """Generate a comprehensive summary report"""
    summary = {
        'benchmark_info': {
            'timestamp': datetime.now().isoformat(),
            'total_models_evaluated': len(all_reports),
            'total_questions': len(mcq_data) + len(tf_data),
            'mcq_questions': len(mcq_data),
            'tf_questions': len(tf_data),
            'total_benchmark_time_minutes': round(total_benchmark_time / 60, 2),
            'config': {
                'max_input_length': MAX_INPUT_LENGTH,
                'max_new_tokens': MAX_NEW_TOKENS,
                'batch_size': BATCH_SIZE,
            }
        },
        'model_comparison': [],
        'ranking': {
            'by_overall_accuracy': [],
            'by_mcq_accuracy': [],
            'by_tf_accuracy': [],
            'by_speed': [],
        }
    }
    
    # Extract comparison data
    for report in all_reports:
        comparison_entry = {
            'model_name': report['model_name'],
            'model_type': report['model_type'],
            'overall_accuracy': report['metrics']['accuracy_percent'],
            'mcq_accuracy': report['metrics']['mcq_accuracy_percent'],
            'tf_accuracy': report['metrics']['tf_accuracy_percent'],
            'null_prediction_rate': report['metrics']['null_prediction_rate_percent'],
            'avg_response_length': report['metrics']['average_response_length'],
            'questions_per_second': report['timing']['questions_per_second'],
            'total_time_seconds': report['timing']['total_time_seconds'],
        }
        summary['model_comparison'].append(comparison_entry)
    
    # Generate rankings
    summary['ranking']['by_overall_accuracy'] = sorted(
        [(r['model_name'], r['metrics']['accuracy_percent']) for r in all_reports],
        key=lambda x: x[1], reverse=True
    )
    summary['ranking']['by_mcq_accuracy'] = sorted(
        [(r['model_name'], r['metrics']['mcq_accuracy_percent']) for r in all_reports],
        key=lambda x: x[1], reverse=True
    )
    summary['ranking']['by_tf_accuracy'] = sorted(
        [(r['model_name'], r['metrics']['tf_accuracy_percent']) for r in all_reports],
        key=lambda x: x[1], reverse=True
    )
    summary['ranking']['by_speed'] = sorted(
        [(r['model_name'], r['timing']['questions_per_second']) for r in all_reports],
        key=lambda x: x[1], reverse=True
    )
    
    return summary

# Generate and save summary
summary_report = generate_summary_report(all_model_reports)

# Save summary report
summary_path = os.path.join(OUTPUT_DIR, "benchmark_summary.json")
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_report, f, ensure_ascii=False, indent=2)
print(f"\nSummary saved: {summary_path}")

# Save all detailed reports
all_reports_path = os.path.join(OUTPUT_DIR, "all_models_detailed_report.json")
with open(all_reports_path, 'w', encoding='utf-8') as f:
    json.dump(all_model_reports, f, ensure_ascii=False, indent=2)
print(f"All reports saved: {all_reports_path}")


Summary saved: ./results/benchmark_summary.json
All reports saved: ./results/all_models_detailed_report.json


In [11]:
# Cell 11: Print Final Summary
print("\n" + "="*70)
print("BENCHMARK RESULTS SUMMARY")
print("="*70)

print(f"\n{'Model':<40} {'Overall%':>10} {'MCQ%':>8} {'T/F%':>8} {'Q/s':>8}")
print("-"*70)

for report in all_model_reports:
    m = report['metrics']
    t = report['timing']
    print(f"{report['model_name']:<40} {m['accuracy_percent']:>10.2f} {m['mcq_accuracy_percent']:>8.2f} {m['tf_accuracy_percent']:>8.2f} {t['questions_per_second']:>8.2f}")

print("\n" + "="*70)
print("RANKINGS")
print("="*70)

print("\nBy Overall Accuracy:")
for i, (name, acc) in enumerate(summary_report['ranking']['by_overall_accuracy'], 1):
    print(f"   {i}. {name}: {acc:.2f}%")

print("\nBy MCQ Accuracy:")
for i, (name, acc) in enumerate(summary_report['ranking']['by_mcq_accuracy'], 1):
    print(f"   {i}. {name}: {acc:.2f}%")

print("\nBy True/False Accuracy:")
for i, (name, acc) in enumerate(summary_report['ranking']['by_tf_accuracy'], 1):
    print(f"   {i}. {name}: {acc:.2f}%")

print("\nBy Speed (questions/second):")
for i, (name, speed) in enumerate(summary_report['ranking']['by_speed'], 1):
    print(f"   {i}. {name}: {speed:.2f} q/s")

print("\n" + "="*70)
print(f"Total benchmark time: {total_benchmark_time/60:.1f} minutes")
print(f"Output files saved in: {OUTPUT_DIR}")
print("="*70)


BENCHMARK RESULTS SUMMARY

Model                                      Overall%     MCQ%     T/F%      Q/s
----------------------------------------------------------------------
Qwen/Qwen3-4B                                 49.90    41.80    58.40     0.20
Qwen/Qwen3-8B                                 51.70    48.44    55.12     0.14
deepseek-ai/deepseek-llm-7b-chat              25.60    30.27    20.70     0.14
keepitreal/vietnamese-sbert                   39.60    29.69    50.00    45.42

RANKINGS

By Overall Accuracy:
   1. Qwen/Qwen3-8B: 51.70%
   2. Qwen/Qwen3-4B: 49.90%
   3. keepitreal/vietnamese-sbert: 39.60%
   4. deepseek-ai/deepseek-llm-7b-chat: 25.60%

By MCQ Accuracy:
   1. Qwen/Qwen3-8B: 48.44%
   2. Qwen/Qwen3-4B: 41.80%
   3. deepseek-ai/deepseek-llm-7b-chat: 30.27%
   4. keepitreal/vietnamese-sbert: 29.69%

By True/False Accuracy:
   1. Qwen/Qwen3-4B: 58.40%
   2. Qwen/Qwen3-8B: 55.12%
   3. keepitreal/vietnamese-sbert: 50.00%
   4. deepseek-ai/deepseek-llm-7b-chat: 20.

In [12]:
# Cell 12: Error Analysis (Optional)
def analyze_errors(report: Dict) -> Dict:
    """Analyze common error patterns for a model"""
    errors = [r for r in report['detailed_results'] if not r['is_correct']]
    
    analysis = {
        'total_errors': len(errors),
        'mcq_errors': len([e for e in errors if e['question_type'] == 'MCQ']),
        'tf_errors': len([e for e in errors if e['question_type'] == 'T/F']),
        'null_answer_errors': len([e for e in errors if e['model_choice'] is None]),
        'sample_errors': errors[:10]  # First 5 errors for inspection
    }
    
    # Analyze answer distribution in errors
    mcq_error_dist = {}
    for e in errors:
        if e['question_type'] == 'MCQ' and e['model_choice']:
            pred = e['model_choice']
            correct = e['correct_answer']
            key = f"{pred} (pred) vs {correct} (correct)"
            mcq_error_dist[key] = mcq_error_dist.get(key, 0) + 1
    
    analysis['mcq_error_distribution'] = mcq_error_dist
    
    return analysis

# Analyze errors for each model
print("\n" + "="*70)
print("ERROR ANALYSIS")
print("="*70)

for report in all_model_reports:
    print(f"\n📋 {report['model_name']}")
    analysis = analyze_errors(report)
    print(f"   Total errors: {analysis['total_errors']}")
    print(f"   MCQ errors: {analysis['mcq_errors']}")
    print(f"   T/F errors: {analysis['tf_errors']}")
    print(f"   Null answer errors: {analysis['null_answer_errors']}")
    
    # Save error analysis
    model_safe_name = report['model_name'].replace('/', '_').replace('-', '_')
    error_path = os.path.join(OUTPUT_DIR, f"error_analysis_{model_safe_name}.json")
    with open(error_path, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, ensure_ascii=False, indent=2)
    print(f"   Error analysis saved: {error_path}")


ERROR ANALYSIS

📋 Qwen/Qwen3-4B
   Total errors: 501
   MCQ errors: 298
   T/F errors: 203
   Null answer errors: 0
   Error analysis saved: ./results/error_analysis_Qwen_Qwen3_4B.json

📋 Qwen/Qwen3-8B
   Total errors: 483
   MCQ errors: 264
   T/F errors: 219
   Null answer errors: 0
   Error analysis saved: ./results/error_analysis_Qwen_Qwen3_8B.json

📋 deepseek-ai/deepseek-llm-7b-chat
   Total errors: 744
   MCQ errors: 357
   T/F errors: 387
   Null answer errors: 324
   Error analysis saved: ./results/error_analysis_deepseek_ai_deepseek_llm_7b_chat.json

📋 keepitreal/vietnamese-sbert
   Total errors: 604
   MCQ errors: 360
   T/F errors: 244
   Null answer errors: 0
   Error analysis saved: ./results/error_analysis_keepitreal_vietnamese_sbert.json
